# MouserCV Keypoint Detection Evaluation — DLC SuperAnimal-Quadruped on Side-View Cage Video

Evaluates DLC SuperAnimal-Quadruped zero-shot keypoint detection quality
on real Bauer lab videos (EGFR pruritus mouse model). Specifically tests
whether paw keypoints are reliable enough for scratch/groom discrimination.

**Key questions:**
1. Which keypoints are reliably detected in side-view cage footage?
2. Are paw keypoints (front + hind) detected above pcutoff consistently?
3. Can paw velocity differentiate scratching from grooming?
4. Is hind paw elevation visible during scratch bouts?

**Runtime:** Google Colab with T4 GPU, or local with CUDA.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--pre",
        "deeplabcut", "opencv-python-headless",
    ])
    print("Install complete — Runtime → Restart session → Run all")
else:
    print("Local: uv venv active — no install needed")


In [ ]:
# ── Google Colab: mount Google Drive ───────────────────────────────────────
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Mounted → /content/drive/MyDrive")
else:
    print("Local — Drive mount skipped")


In [1]:
import os

# === CONFIGURATION ===
# Path to video (adjust for Colab: upload or mount GDrive)
# ── Video path: GDrive (Colab) vs local ─────────────────────────────────
GDRIVE_VIDEO_DIR = "/content/drive/MyDrive/mousercv"  # ← adjust to your GDrive folder
VIDEO_PATH = (
    f"{GDRIVE_VIDEO_DIR}/Cage 17082 video.MOV"
    if IN_COLAB
    else "../data/videos/CV analysis project/Cage 17082 video.MOV"
)

# Time range to evaluate
START_TIME = "13:50"
END_TIME = "14:05"

# DLC model
SUPERANIMAL_NAME = "superanimal_quadruped"
MODEL_NAME = "hrnet_w32"
DETECTOR_NAME = "fasterrcnn_resnet50_fpn_v2"

# Confidence threshold for "reliable" keypoints
PCUTOFF = 0.3

# Essential keypoints for behavior discrimination
ESSENTIAL_KEYPOINTS = [
    "nose",
    "front_left_paw", "front_right_paw",  # grooming
    "back_left_paw", "back_right_paw",    # scratching
    "tail_base",                           # body orientation
]

# All 39 SuperAnimal-Quadruped keypoints for reference
ALL_KEYPOINTS = [
    "nose", "upper_jaw", "lower_jaw", "mouth_end_right", "mouth_end_left",
    "right_eye", "right_earbase", "right_earend", "right_antler_base", "right_antler_end",
    "left_eye", "left_earbase", "left_earend", "left_antler_base", "left_antler_end",
    "neck_base", "neck_end", "throat_base", "throat_end",
    "back_base", "back_end", "back_middle",
    "tail_base", "tail_end",
    "front_left_thai", "front_left_knee", "front_left_paw",
    "front_right_thigh", "front_right_knee", "front_right_paw",
    "back_left_paw", "back_left_thigh",
    "back_right_thai", "back_left_knee", "back_right_knee", "back_right_paw",
    "belly_bottom", "body_middle_right", "body_middle_left",
]

In [2]:
def time_to_seconds(time_str: str) -> float:
    """Convert mm:ss or mm:ss.ms or hh:mm:ss to seconds."""
    parts = time_str.strip().split(":")
    if len(parts) == 2:
        return float(parts[0]) * 60 + float(parts[1])
    elif len(parts) == 3:
        return float(parts[0]) * 3600 + float(parts[1]) * 60 + float(parts[2])
    return float(time_str)


def seconds_to_time(seconds: float) -> str:
    """Convert seconds to mm:ss.ms format."""
    m, s = divmod(seconds, 60)
    return f"{int(m)}:{s:05.2f}"


def time_to_frame(time_str: str, fps: float) -> int:
    """Convert time string to frame number."""
    return int(time_to_seconds(time_str) * fps)


def frame_to_time(frame: int, fps: float) -> str:
    """Convert frame number to time string."""
    return seconds_to_time(frame / fps)

In [3]:
import cv2
import numpy as np

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

start_frame = time_to_frame(START_TIME, fps)
end_frame = time_to_frame(END_TIME, fps)

cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
frames = []
for i in range(end_frame - start_frame):
    ret, frame = cap.read()
    if not ret:
        break
    frames.append(frame)
cap.release()

print(f"Extracted {len(frames)} frames ({len(frames)/fps:.1f}s) from {os.path.basename(VIDEO_PATH)}")
print(f"Resolution: {width}x{height}, FPS: {fps}")

Extracted 1800 frames (60.0s) from Cage 2.MOV
Resolution: 1920x1080, FPS: 30.001825805356336


In [ ]:
import deeplabcut

# Save frames as temp video for DLC processing
TEMP_VIDEO = "/tmp/mousercv_eval_clip.mp4"
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(TEMP_VIDEO, fourcc, fps, (width, height))
for frame in frames:
    out.write(frame)
out.release()
print(f"Saved {len(frames)} frames as temp video")

# Run SuperAnimal inference — capture the returned dict of DataFrames
print("Running DLC SuperAnimal-Quadruped inference...")
results = deeplabcut.video_inference_superanimal(
    [TEMP_VIDEO],
    superanimal_name=SUPERANIMAL_NAME,
    model_name=MODEL_NAME,
    detector_name=DETECTOR_NAME,
    video_adapt=False,
    pcutoff=PCUTOFF,
)
print("Inference complete!")

In [ ]:
import pandas as pd

# DLC 3.0 returns results as a dict: {video_path: DataFrame}
# The DataFrame has a 4-level MultiIndex: (scorer, individuals, bodyparts, coords)
df_multi = list(results.values())[0]

scorer = df_multi.columns.get_level_values("scorer")[0]
individuals = df_multi.columns.get_level_values("individuals").unique()
keypoint_names = df_multi.columns.get_level_values("bodyparts").unique()

print(f"Shape       : {df_multi.shape}")
print(f"Scorer      : {scorer}")
print(f"Individuals : {list(individuals)}")
print(f"Keypoints ({len(keypoint_names)}): {list(keypoint_names)}")

# Select the best individual per frame (highest mean likelihood)
# For single-mouse cage video, this is typically individual_0
mean_conf = {}
for ind in individuals:
    liks = df_multi.loc[:, (scorer, ind, slice(None), "likelihood")]
    mean_conf[ind] = liks.mean(axis=1).mean()
best_individual = max(mean_conf, key=mean_conf.get)
print(f"\nBest individual: {best_individual} (mean conf {mean_conf[best_individual]:.3f})")

# Extract a flat (scorer, bodypart, coord) DataFrame for the best individual
df = df_multi.loc[:, (scorer, best_individual)]
# Rebuild with 3-level index so annotation cell can index as (scorer, kp, coord)
df.columns = pd.MultiIndex.from_tuples(
    [(scorer, bp, c) for bp, c in df.columns],
    names=["scorer", "bodyparts", "coords"],
)
print(f"Working DataFrame: {df.shape} — {len(keypoint_names)} keypoints × {len(df)} frames")

In [ ]:
import subprocess
from IPython.display import Video, display

# === CONFIGURE — pick any sub-range within START_TIME..END_TIME ===
PLAY_START   = "13:53"   # mm:ss
PLAY_END     = "14:00"   # mm:ss
OUTPUT_SCALE = 0.5      # 1.0 = full 1920x1080; 0.5 = 960x540 (faster, smaller)

# ── resolve to indices in the already-loaded `frames` list ─────────────────
p0 = max(0,           time_to_frame(PLAY_START, fps) - start_frame)
p1 = min(len(frames), time_to_frame(PLAY_END,   fps) - start_frame)
out_w, out_h = int(width * OUTPUT_SCALE), int(height * OUTPUT_SCALE)
print(f"{PLAY_START} → {PLAY_END}  |  {p1-p0} frames ({(p1-p0)/fps:.1f}s)  |  {out_w}×{out_h}")

# ── write annotated video (mp4v first, then re-encode to H.264) ───────────
tmp_path = "/tmp/mousercv_annotated_raw.mp4"
out_path = "/tmp/mousercv_annotated.mp4"
writer = cv2.VideoWriter(tmp_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (out_w, out_h))

for fi in range(p0, p1):
    frame = frames[fi].copy()
    abs_f = start_frame + fi

    for kp in keypoint_names:
        try:
            x    = df.iloc[fi][(scorer, kp, "x")]
            y    = df.iloc[fi][(scorer, kp, "y")]
            conf = df.iloc[fi][(scorer, kp, "likelihood")]
            if conf > PCUTOFF:
                color  = (0, 220, 0) if kp in ESSENTIAL_KEYPOINTS else (160, 160, 160)
                radius = 9           if kp in ESSENTIAL_KEYPOINTS else 4
                cv2.circle(frame, (int(x), int(y)), radius, color, -1)
                if kp in ESSENTIAL_KEYPOINTS:
                    cv2.putText(frame, kp.split("_")[-1],
                                (int(x) + 10, int(y) - 8),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)
        except (KeyError, IndexError):
            pass

    cv2.putText(frame, frame_to_time(abs_f, fps),
                (18, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (255, 255, 255), 3, cv2.LINE_AA)
    writer.write(cv2.resize(frame, (out_w, out_h), interpolation=cv2.INTER_AREA))

writer.release()

# Re-encode to H.264 so the video plays natively in Jupyter / browsers
subprocess.run(
    ["ffmpeg", "-y", "-i", tmp_path, "-c:v", "libx264", "-preset", "fast",
     "-crf", "23", "-pix_fmt", "yuv420p", "-an", out_path],
    capture_output=True, check=True,
)
print(f"Saved → {out_path}")
display(Video(out_path, embed=True, width=960))